# Masking into MERRA-2 Datasets

A little notebook to showcase how I might start using my catalog to mask into MERRA-2

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import os
from pathlib import Path
import seaborn as sns
import xarray as xr
from tqdm import tqdm
import dask

import h5py
import requests

import boto3
import s3fs
import earthaccess
from IPython.display import display, Markdown
from ipywidgets import IntProgress

repo_dir = str(Path(os.getcwd()).parents[0])
os.chdir(repo_dir + '/scripts/')
from utils import display_catalog

home_dir = str(Path(os.getcwd()).parents[0])

## Setting Up MERRA-2 Streaming

In [2]:
if (boto3.client('s3').meta.region_name == 'us-west-2'):
    display(Markdown('### us-west-2 Region Check: &#x2705;'))
else:
    display(Markdown('### us-west-2 Region Check: &#10060;'))
    raise ValueError('Your notebook is not running inside the AWS us-west-2 region, and will not be able to directly access NASA Earthdata S3 buckets')

### us-west-2 Region Check: &#x2705;

## Loading Up the Catalog

In [3]:
# load up all of the dataframes by year, and then concatenate into one big one
df_path = home_dir + '/data/ar_database/dataframes/'
fnames = os.listdir(df_path)
df_list = []

for fname in fnames:
    df_list.append(pd.read_hdf(df_path + fname))
    
dataframe = pd.concat(df_list)

## Storm Characteristics

Adding storm characteristics to the database using MERRA-2 data.

### Areas and Durations

In [4]:
# load up relevant MERRA-2 datasets
grid_areas = xr.open_dataset(home_dir + '/data/area/MERRA2_gridarea.nc')
grid_areas = grid_areas.sel(lat=slice(-86, -39)).cell_area

In [5]:
def compute_max_area(ar_da):
    grid_area_storm = grid_areas.sel(lat=ar_da.lat, lon=ar_da.lon)
    max_area = float(ar_da.dot(grid_area_storm).max().values/(1000**2))
    return max_area

def compute_mean_area(ar_da):
    grid_area_storm = grid_areas.sel(lat=ar_da.lat, lon=ar_da.lon)
    mean_area = float(ar_da.dot(grid_area_storm).mean().values/(1000**2))
    return mean_area

def compute_duration(ar_da):
    days = (ar_da.time.max() - ar_da.time.min()).values.astype('timedelta64[h]').astype(int) + np.timedelta64(3, 'h')
    return days

def add_start_date(ar_da):
    start = ar_da.time.min().values
    return start

def add_end_date(ar_da):
    end = ar_da.time.max().values
    return end

In [6]:
dataframe['max_area'] = dataframe['data_array'].apply(compute_max_area)
dataframe['mean_area'] = dataframe['data_array'].apply(compute_mean_area)
dataframe['duration'] = dataframe['data_array'].apply(compute_duration)
dataframe['start_date'] = dataframe['data_array'].apply(add_start_date)
dataframe['end_date'] = dataframe['data_array'].apply(add_end_date)

In [7]:
attr_df = dataframe[['max_area', 'mean_area', 'duration', 'start_date', 'end_date', 'is_landfalling']]

In [8]:
attr_df = attr_df.sort_values(by='start_date')

### Atmospheric Variables

In [24]:
cell_areas = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/area/MERRA2_gridarea.nc')
cell_areas = cell_areas.cell_area
ais_mask = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/antarctic_masks/AIS_Full_basins_Zwally_MERRA2grid_new.nc')
ais_mask = ais_mask > 0

In [49]:
def compute_cumulative(storm_da, var_da, area_da, ais_da=None):
    if ais_da is not None:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()

    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)

    storm_cell_areas = area_da.sel(lat=storm_da.lat, lon=storm_da.lon)
    avg_storm_val = float(storm_cell_areas.dot(storm_da_subset*var_da_subset).mean())

    return avg_storm_val

def compute_max_intensity(storm_da, var_da, area_da, ais_da=None):
    if ais_da is not None:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()
        
    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)
    max_intensity_val = float((storm_da_subset*var_da_subset).max())

    return max_intensity_val

def compute_min_intensity(storm_da, var_da, area_da, ais_da=None):
    if ais_da is not None:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()
        
    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)
    max_intensity_val = float((storm_da_subset*var_da_subset).min())

    return max_intensity_val

def compute_average(storm_da, var_da, area_da, ais_da=None):
    if ais_da is not None:
        storm_ais_mask = ais_da.sel(lat=storm_da.lat, lon=storm_da.lon).Zwallybasins
        storm_da_subset = storm_da.where(storm_ais_mask, 0)
    else:
        storm_da_subset = storm_da.copy()

    var_da_subset = var_da.sel(lat=storm_da_subset.lat, lon=storm_da_subset.lon)

    storm_cell_areas = area_da.sel(lat=storm_da.lat, lon=storm_da.lon)
    tot_area = storm_da_subset.dot(storm_cell_areas)
    avg_storm_val = float((storm_cell_areas.dot(storm_da_subset*var_da_subset)/tot_area).mean())

    return avg_storm_val

#### 2m-Temperature, 10m Poleward Wind, Total Precipitable Vapor

In [50]:
gesdisc_s3 = "https://data.gesdisc.earthdata.nasa.gov/s3credentials"

# Define a function for S3 access credentials

def begin_s3_direct_access(url: str=gesdisc_s3):
    response = requests.get(url).json()
    return s3fs.S3FileSystem(key=response['accessKeyId'],
                             secret=response['secretAccessKey'],
                             token=response['sessionToken'],
                             client_kwargs={'region_name':'us-west-2'})

fs = begin_s3_direct_access()

In [51]:
def grab_MERRA2_granules(storm_da, data_doi):

    first = np.min(storm_da.time.dt.date.to_numpy())
    last = np.max(storm_da.time.dt.date.to_numpy())
    # stream the data only between those two dates
    results = earthaccess.search_data(doi=data_doi, 
                                  temporal=(f'{first.year}-{first.month}-{first.day}', 
                                            f'{last.year}-{last.month}-{last.day}'))

    granule_lst = []
    for result in results:
        link = result.data_links()[0].replace('https://data.gesdisc.earthdata.nasa.gov/data', 's3://gesdisc-cumulus-prod-protected')
        granule = fs.open(link)
        granule_lst.append(granule)

    return granule_lst

def compute_summaries(storm_da, func_vars_dict, cell_areas, data_doi, ais_mask=None):

    granule_lst = grab_MERRA2_granules(storm_da, data_doi)

    ds_lst = []
    for granule in granule_lst:
        ds = xr.open_dataset(granule)
        ds_lst.append(ds[list(func_vars_dict.keys())].sel(lat=slice(-86, -39)).sel(time = ds.time.dt.hour % 3 == 0))

    obs_ds = xr.concat(ds_lst, dim='time')

    summaries = []
    for key, func in func_vars_dict.items():
        single_var_da = obs_ds[key]
        summaries.append(func(storm_da, single_var_da, cell_areas))

    return summaries
    

In [52]:
from dask.distributed import Client, progress
import logging
client = Client(threads_per_worker=2, n_workers=10)
client

/home/jovyan/envs/antarctic_ars/lib/python3.12/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39483 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jbbutler/proxy/39483/status,
Dashboard: /user/jbbutler/proxy/39483/status,Workers: 10
Total threads: 20,Total memory: 14.84 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34865,Workers: 10
Dashboard: /user/jbbutler/proxy/39483/status,Total threads: 20
Started: Just now,Total memory: 14.84 GiB
Comm: tcp://127.0.0.1:36763,Total threads: 2
Dashboard: /user/jbbutler/proxy/45597/status,Memory: 1.48 GiB
Nanny: tcp://127.0.0.1:39241,


In [53]:
data_doi = '10.5067/3Z173KIE2TPD'

cell_areas = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/area/MERRA2_gridarea.nc').cell_area
cell_areas = dask.delayed(cell_areas)
ais_mask = xr.open_dataset('/home/jovyan/extreme_antarctic_ARs/data/antarctic_masks/AIS_Full_basins_Zwally_MERRA2grid_new.nc')
ais_mask = ais_mask > 0
# changing the zero lon coordinate in the ais mask to the same close-to-zero value we see in the other dataarrays
# should probably change this in the future
ais_lons = ais_mask.lon.values
ais_lons[ais_lons == 0] = -5.920304e-13
ais_mask = ais_mask.assign_coords({'lon': ais_lons})
#ais_mask = dask.delayed(ais_mask)

func_vars_dict = {'T2M': lambda storm_da, var_da, area_da: compute_average(storm_da, var_da, area_da, ais_mask),
                  'V10M': lambda storm_da, var_da, area_da: compute_average(storm_da, var_da, area_da, ais_mask), 
                  'SLP': lambda storm_da, var_da, area_da: compute_min_intensity(storm_da, var_da, area_da),
                  'TQV': lambda storm_da, var_da, area_da: compute_average(storm_da, var_da, area_da, ais_mask)}

landfalling_dataframe = dataframe[dataframe.is_landfalling]
landfalling_dataframe = landfalling_dataframe.iloc[0:30]
lazy_results = []

for index, storm in landfalling_dataframe.iterrows():
    
    storm_da = dask.delayed(storm.data_array)
    lazy_result = dask.delayed(compute_summaries)(storm_da, func_vars_dict, cell_areas, data_doi, ais_mask)
    lazy_results.append(lazy_result)


In [ ]:
%%time

results = dask.compute(*lazy_results);

In [134]:
results

([261.95120858886844, -0.853522204227583, 0.0, 5.324114879592379],
 [254.25787529628604, 1.51414761695872, 0.0, 2.303332334543756],
 [267.7986573139681, 3.792203337054628, 0.0, 7.43824160671476],
 [259.3493906584122, 8.400122965652493, 0.0, 6.269893597363971],
 [273.1973364935633, -6.461016173063086, 0.0, 14.506770601531917],
 [274.61995856069285, -5.0988943660971255, 0.0, 17.80057970582737],
 [245.94145457095865, 0.9797684602835641, 0.0, 2.434041559849784],
 [254.23347239149334, 1.26557122773882, 0.0, 2.1562806516640847],
 [254.38078548023032, 1.7761074622440678, 0.0, 4.97477783189922],
 [244.9998710321368, -1.2041460077945532, 0.0, 1.242682622838415],
 [259.9954386813924, 0.9519747111971959, 0.0, 5.43971562309421],
 [273.60538182867253, -6.245808945622117, 0.0, 14.344821212054468],
 [265.54680578877145, -6.007894060816648, 0.0, 8.331370255192741],
 [254.18395199989726, -0.3285768454615112, 0.0, 5.08032851234803],
 [251.9953866005441, 0.13007597937441895, 0.0, 3.4675378766229916],
 [2

### 10m Winds

In [ ]:
start_day = attr_df.start_date.dt.date.iloc[0]
end_day = attr_df.end_date.dt.date.iloc[-1]

In [ ]:
days = pd.date_range(start=start_day, end=end_day).strftime('%Y%m%d')

In [ ]:
days

In [ ]:
# load up the 10m wind data

day = days[15000]

In [ ]:
day

In [ ]:
path

In [ ]:
path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/'+ day[0:4] +'/' + day[4:6] + '/MERRA2_400.inst1_2d_asm_Nx.' + day + '.nc4'
#path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/2022/03/MERRA2_400.inst1_2d_asm_Nx.20220315.nc4'
dat = xr.open_dataset(fs.open(path), decode_cf=True,)

In [ ]:

# Files are organized by s3://gesdisc-cumulus-prod-protected/MERRA2/M2T1NXSLV.5.12.4/year/mo/*.nc4
datasets = []

for day in days:
    
    path = 's3://gesdisc-cumulus-prod-protected/MERRA2/M2I1NXASM.5.12.4/'+ day[0:4] +'/' + day[4:6] + '/MERRA2_400.inst1_2d_asm_Nx.' + day + '.nc4'
    dat = xr.open_dataset(fs.open(path), decode_cf=True,)
    antarctic_dat = dat.sel(lat = slice(-90.0, -50.0), lon = slice(-180, 179.4))
    datasets.append(antarctic_dat[['T2M', 'TQV', 'V10M']])

#### h5coro implementation

In [ ]:
# tried to use h5coro at one point to load up the data granules, instead of using xarray openmfdataset
# seems like it's not much faster at all, and while it could be if I could input multiple days, documentation for the package is very bad
# and I don't feel like reading source code to find a magical function that optimizes the loading of multiple days of data..

# Initialize the H5Coro object with the S3 driver and credentials
#h5obj = h5coro.H5Coro(paths[0].details['name'], h5coro.s3driver.S3Driver, 
                      credentials=s3_creds, errorChecking=True, verbose=False, multiProcess=False)

# Define the variables you want to read
#variables = ['SLP', 'T2M', 'V10M']
# Read the data using h5coro
#data = h5obj.readDatasets(variables, block=True, enableAttributes=False)

#time = pd.date_range(start=pd.Timestamp(start_date), end=pd.Timestamp(end_date)+pd.Timedelta(1, unit='D'), freq='1h', inclusive='left')
#lat = np.arange(-90, 90.5, .5)
#lon = np.arange(-180, 180, 0.625)

In [1]:
import xarray as xr

In [2]:
xr.open_dataset('/pscratch/sd/j/jbbutler/merra2_data_850hPa_wind/inst1_2d_asm_Nx.19970315.nc4.nc4')

<xarray.Dataset> Size: 21MB
Dimensions:  (lon: 576, time: 24, lat: 95)
Coordinates:
  * lon      (lon) float64 5kB -180.0 -179.4 -178.8 -178.1 ... 178.1 178.8 179.4
  * time     (time) datetime64[ns] 192B 1997-03-15 ... 1997-03-15T23:00:00
  * lat      (lat) float64 760B -86.0 -85.5 -85.0 -84.5 ... -40.0 -39.5 -39.0
Data variables:
    T2M      (time, lat, lon) float32 5MB ...
    SLP      (time, lat, lon) float32 5MB ...
    TQV      (time, lat, lon) float32 5MB ...
    V10M     (time, lat, lon) float32 5MB ...
Attributes: (12/37)
    History:                             Original file generated: Fri Oct 24 ...
    Comment:                             GMAO filename: d5124_m2_jan91.inst1_...
    Filename:                            MERRA2_200.inst1_2d_asm_Nx.19970315.nc4
    Conventions:                         CF-1
    Institution:                         NASA Global Modeling and Assimilatio...
    References:                          http://gmao.gsfc.nasa.gov
    ...                                  ...
    build_dmrpp_metadata.bes:            3.21.0-272
    build_dmrpp_metadata.libdap:         libdap-3.21.0-70
    build_dmrpp_metadata.configuration:  \n# TheBESKeys::get_as_config()\nAll...
    build_dmrpp_metadata.invocation:     build_dmrpp -c /tmp/bes_conf_6OHq -f...
    history:                             2024-10-30 01:22:51 GMT hyrax-1.17.0...
    history_json:                        [{"$schema":"https:\/\/harmony.earth...